# Normalization

> Picklable per-column scalers fitted on dataset columns (stable-worldmodel API).

In [ ]:
#| default_exp data.normalization

In [ ]:
#| hide
from nbdev.showdoc import *

Each scaler has `fit` / `transform` / `inverse_transform` / `fit_transform` (the
`Transformable` protocol plus the sklearn pattern) and is callable. Statistics are per entry of
the last axis, over every other axis: for a per-agent column ``(N, num_agents, D)`` the agents
are pooled; for ``action`` ``(N, num_agents)`` each agent gets its own. Rows with a ``nan``
(e.g. the action of an episode's last row) are left out of the fit. Inputs may be numpy arrays or
torch tensors, and the output matches (as floats: integer columns, e.g. positions, are converted). `ColumnTransform` applies one to a dataset column, as
a dataset ``transform``.

In [ ]:
#| export
# Adapted from stable-worldmodel's data package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from typing import Any

import numpy as np

In [ ]:
#| export
def _as_float(x):
    "`x` as floats (integer columns such as positions would otherwise truncate the statistics)."
    if _is_tensor(x):
        return x if x.is_floating_point() else x.float()
    x = np.asarray(x)
    return x if x.dtype.kind == 'f' else x.astype(np.float32)


def _as_like(a, b, x):
    "Bounds `a`, `b` as the type (and dtype / device) of `x`."
    if _is_tensor(x):
        import torch
        return torch.as_tensor(a, dtype=x.dtype, device=x.device), torch.as_tensor(b, dtype=x.dtype, device=x.device)
    return np.asarray(a), np.asarray(b)


def _is_tensor(x) -> bool:
    return type(x).__module__.startswith('torch')


def _to_numpy_2d(X) -> np.ndarray:
    "``(N, D)`` rows of `X` (last axis kept), without rows holding a nan."
    arr = X.detach().cpu().numpy() if _is_tensor(X) else np.asarray(X)
    arr = arr.reshape(-1, arr.shape[-1]) if arr.ndim > 1 else arr.reshape(-1, 1)
    return arr[~np.isnan(arr).any(axis=1)]


class IdentityScaler:
    "Leaves values unchanged."
    def fit(self, X): return self
    def transform(self, X): return X
    def inverse_transform(self, X): return X
    def fit_transform(self, X): return X
    def __call__(self, X): return X


class ZScoreScaler:
    "``(x - mean) / std`` per entry of the last axis (statistics kept as numpy arrays, so it pickles)."
    def __init__(self, mean=None, std=None, eps: float = 1e-8):
        self.mean = np.asarray(mean) if mean is not None else None
        self.std = np.asarray(std) if std is not None else None
        self.eps = eps

    def fit(self, X):
        data = _to_numpy_2d(X)
        self.mean, self.std = data.mean(axis=0, keepdims=True), data.std(axis=0, keepdims=True)
        return self

    def transform(self, X):
        X = _as_float(X)
        mean, std = _as_like(self.mean, self.std, X)
        return (X - mean) / (std.clamp(min=self.eps) if _is_tensor(X) else np.maximum(std, self.eps))

    def inverse_transform(self, X):
        X = _as_float(X)
        mean, std = _as_like(self.mean, self.std, X)
        return X * std + mean

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def __call__(self, X):
        out = self.transform(X)
        return out.float() if _is_tensor(out) else out


class PercentileScaler:
    "Maps ``[q_low, q_high]`` (the `low` / `high` percentiles) to ``[-1, 1]``, clipped: robust to outliers."
    def __init__(self, low: float = 1.0, high: float = 99.0, q_low=None, q_high=None, eps: float = 1e-8):
        self.low, self.high, self.eps = low, high, eps
        self.q_low = np.asarray(q_low) if q_low is not None else None
        self.q_high = np.asarray(q_high) if q_high is not None else None

    def fit(self, X):
        data = _to_numpy_2d(X)
        self.q_low, self.q_high = np.percentile(data, self.low, axis=0), np.percentile(data, self.high, axis=0)
        return self

    def transform(self, X):
        X = _as_float(X)
        q_low, q_high = _as_like(self.q_low, self.q_high, X)
        if _is_tensor(X):
            return (2 * (X - q_low) / (q_high - q_low).clamp(min=self.eps) - 1).clamp(-1, 1)
        return np.clip(2 * (X - q_low) / np.maximum(q_high - q_low, self.eps) - 1, -1, 1)

    def inverse_transform(self, X):
        X = _as_float(X)
        q_low, q_high = _as_like(self.q_low, self.q_high, X)
        return (X + 1) * (q_high - q_low) / 2 + q_low

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def __call__(self, X):
        out = self.transform(X)
        return out.float() if _is_tensor(out) else out


SCALERS = {'zscore': ZScoreScaler, 'percentile': PercentileScaler, 'none': IdentityScaler}


def get_scaler(method: str = 'zscore', **kwargs: Any):
    "An unfitted scaler: ``'zscore'``, ``'percentile'`` or ``'none'``."
    if method not in SCALERS:
        raise ValueError(f"Unknown normalizer method: {method!r}. Expected one of {list(SCALERS)}.")
    return SCALERS[method](**kwargs)


class ColumnTransform:
    """
    Dataset ``transform`` applying `fn` to the item's `source` column, stored as `target`
    (stable-pretraining's ``WrapTorchTransform``, without the dependency). Picklable.
    """
    def __init__(self, fn, source: str, target: str | None = None):
        self.fn, self.source, self.target = fn, source, target or source

    def __call__(self, item: dict) -> dict:
        if self.source in item:
            item[self.target] = self.fn(item[self.source])
        return item

    def __repr__(self) -> str:
        return f"ColumnTransform({type(self.fn).__name__}, {self.source!r} -> {self.target!r})"

### Tests

In [ ]:
import pickle, torch
from fastcore.test import test_fail

X = np.random.default_rng(0).normal(3.0, 2.0, size=(1000, 2, 3))        # (N, agents, D)
X[5] = np.nan                                                           # left out of the fit
z = ZScoreScaler().fit(X)
assert z.mean.shape == (1, 3) and np.allclose(z.mean, 3.0, atol=0.2) and np.allclose(z.std, 2.0, atol=0.2)
Y = z(torch.as_tensor(X[:10]))
assert Y.dtype == torch.float32 and torch.allclose(torch.as_tensor(z.inverse_transform(z.transform(X[:5]))), torch.as_tensor(X[:5]))
p = PercentileScaler().fit(X)
assert np.nanmax(np.abs(p(X))) <= 1.0 and np.allclose(p.inverse_transform(p.transform(X[:3])), np.clip(X[:3], p.q_low, p.q_high))
actions = np.array([[0., 1.], [2., 3.], [np.nan, np.nan]])             # (N, agents): one statistic per agent
assert ZScoreScaler().fit(actions).mean.tolist() == [[1.0, 2.0]]
assert get_scaler('none')(X) is X and isinstance(get_scaler('percentile', low=5), PercentileScaler)
test_fail(lambda: get_scaler('minmax'), contains='Unknown normalizer')
t = pickle.loads(pickle.dumps(ColumnTransform(z, 'position', 'position_norm')))
item = t({'position': torch.as_tensor(X[:4])})
assert set(item) == {'position', 'position_norm'} and torch.allclose(item['position_norm'], z(torch.as_tensor(X[:4])))
# integer columns (e.g. positions) are normalized as floats, not truncated
pos = torch.tensor([[1, 2], [3, 4], [5, 6]])
z = ZScoreScaler().fit(pos)
assert torch.allclose(z(pos), torch.tensor([[-1.2247, -1.2247], [0.0, 0.0], [1.2247, 1.2247]]), atol=1e-4)
assert np.allclose(PercentileScaler(low=0, high=100).fit(pos.numpy())(pos.numpy())[:, 0], [-1, 0, 1])


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()